# 第 2 天：理解 LSTM 网络

“30 天读 30 篇论文”的第 2 天。

今天学习**长短期记忆（Long Short-Term Memory，LSTM）网络**，序列建模中的一项重要突破。LSTM 通过门控和细胞状态缓解基本 RNN 的梯度消失问题，帮助模型学习长程依赖。

## 本节内容

1. **问题**：基本 RNN 为什么难以处理长序列
2. **思路**：LSTM 如何通过门控控制信息流
3. **架构**：四个主要组成部分——遗忘门、输入门、候选细胞状态、输出门
4. **实现**：使用 NumPy 从零构建 LSTM
5. **可视化**：观察 LSTM “记住”和“遗忘”什么

## Colah 原文的核心认识

细胞状态与隐藏状态并行传递，像一条传送带（Colah 的比喻）。三个门——遗忘门、输入门和输出门——控制信息的修改与读取；沿直接路径的信息可以在合适的门值下保留。

> **原仓库接口问题：** 本 Notebook 的构造函数调用使用 `input_size`、`output_size`，与当前 `implementation.py` 的 `vocab_size`、`hidden_size`、`seq_length` 接口不一致。后续的 `forward`、`backward`、`sample` 调用及状态属性也存在不一致，原样执行会在创建模型时失败。汉化保留代码语义并明确记录问题，不把此 Notebook 声称为已通过完整执行的教程。梯度曲线是标量衰减示意，不是实测模型梯度。已有输出保留为历史记录，不代表本次验证结果。

In [ ]:
# 设置
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将父目录加入导入路径
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

# 导入本仓库的 LSTM 实现
from implementation import LSTM
from visualization import (
    plot_gate_activations, 
    plot_cell_state_evolution,
    plot_gradient_flow_comparison,
    analyze_gate_patterns
)

# 设置随机种子，以便复现
np.random.seed(42)

print('所有导入成功。')
print(f"NumPy 版本：{np.__version__}")

## 1. 梯度消失问题

基本循环神经网络（vanilla RNN）可以学习一些短程模式，但学习长程依赖较困难。

### 为什么

在随时间反向传播中，梯度按如下形式传递：

$$\frac{\partial L}{\partial h_1} = \frac{\partial L}{\partial h_T} \prod_{t=2}^{T} \frac{\partial h_t}{\partial h_{t-1}}$$

如果沿某一方向的连乘因子通常小于 1，例如每步为 0.9：

$$0.9 \times 0.9 \times 0.9 \times ... \times 0.9 \text{ (50 times)} \approx 0.005$$

梯度便会显著衰减，网络难以学到 50 步之前的信息。这里是直观简化，实际梯度涉及矩阵雅可比和激活导数。

Colah 原文的例子：在 “the clouds are in the ___”（云在___中）中预测 “sky” 比较容易，因为依赖较短；在 “I grew up in France... I speak fluent ___”（我在法国长大……我能流利地说___）中预测 “French” 则需要长程信息。

In [ ]:
# 演示梯度消失
def simulate_gradient_flow(initial_grad=1.0, steps=50, factor=0.9):
    '模拟梯度沿时间反向传递。'
    gradients = [initial_grad]
    for _ in range(steps):
        gradients.append(gradients[-1] * factor)
    return gradients

# 比较不同情形
steps = range(51)
vanilla_rnn = simulate_gradient_flow(1.0, 50, 0.9)
lstm_sim = simulate_gradient_flow(1.0, 50, 0.99)  # 用较小衰减因子示意梯度保留，并非模型实测

plt.figure(figsize=(12, 5))
plt.plot(steps, vanilla_rnn, 'r-', linewidth=2, label='基本 RNN 示意（因子 0.9）')
plt.plot(steps, lstm_sim, 'g-', linewidth=2, label='LSTM 示意（因子 0.99）')
plt.axhline(y=0.1, color='orange', linestyle='--', alpha=0.5, label='衰减参考阈值')
plt.xlabel('反向时间步数', fontsize=12)
plt.ylabel('梯度量级', fontsize=12)
plt.title('梯度衰减示意：RNN 与 LSTM', fontsize=14, fontweight='bold')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"基本 RNN 示意，50 步后：{vanilla_rnn[-1]:.6f}")
print(f"LSTM 示意，50 步后：{lstm_sim[-1]:.6f}")
print(f"\nLSTM 示意保留的梯度为前者的 {lstm_sim[-1] / vanilla_rnn[-1]:.1f} 倍！")

## 2. LSTM 的思路：细胞状态

LSTM 增加**细胞状态（cell state）**，提供与隐藏状态并行的一条路径，帮助缓解梯度消失。

Colah 将细胞状态比作“传送带”：只要 LSTM 不通过门控主动修改信息，它就可以沿着这条路径保留。

### 关键方程

细胞状态通过保留项与写入项的**加性更新**计算：

$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

固定门值时，沿细胞状态直接路径的反向传播贡献为：

$$\frac{\partial C_t}{\partial C_{t-1}} = f_t$$

当模型学到 $f_t \approx 1$ 时，梯度就能沿这条路径较好地保留。这是 LSTM 有效的重要原因。原文将其简化为“梯度不会消失”，但这并非无条件保证，也不是完整计算图的总导数。

## 3. LSTM 架构：四个组成部分

按照 Colah 的逐步讲解，LSTM 单元包含四个主要组成部分：

### 1. 遗忘门（$f_t$）：从细胞状态中丢弃什么

$$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$

- 输出位于 0（全部遗忘）到 1（全部保留）之间。
- Colah 的例子：遇到新主语时，遗忘旧主语的性别。

### 2. 输入门（$i_t$）：存储哪些新信息

$$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$$

控制写入多少新的候选值。

### 3. 候选细胞状态（$\tilde{C}_t$）：可能写入的新值

$$\tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)$$

输出位于 -1 到 1 之间。这些候选值先由输入门缩放，再加入细胞状态。

### 4. 输出门（$o_t$）：从细胞状态输出什么

$$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$$

Colah 的例子：刚遇到主语后，输出其单复数信息，以便选择正确的动词形式。

### 更新过程

1. **遗忘**：$C_t = f_t \odot C_{t-1}$（缩放旧信息）。
2. **写入**：$C_t = C_t + i_t \odot \tilde{C}_t$（加入新信息）。
3. **输出**：$h_t = o_t \odot \tanh(C_t)$（筛选对外暴露的信息）。

细胞状态（$C_t$）承载记忆，隐藏状态（$h_t$）用于输出。

In [ ]:
# 创建小词表和 LSTM，用于可视化
chars = list("hello")
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}
vocab_size = len(chars)

print(f"词表：{chars}")
print(f"词表大小：{vocab_size}")

# 初始化小型 LSTM
hidden_size = 10  # 较小维度便于可视化
lstm = LSTM(input_size=vocab_size, hidden_size=hidden_size, output_size=vocab_size)

print(f"\n已创建 LSTM：")
print(f"  输入维度：{vocab_size}")
print(f"  隐藏状态维度：{hidden_size}")
print(f"  输出维度：{vocab_size}")
print(f"  参数总数：{sum(p.size for p in [lstm.Wf, lstm.Wi, lstm.Wc, lstm.Wo, lstm.Wy])}")

## 4. 前向传播：观察各门

将序列 “hello” 输入 LSTM，记录每一步的遗忘门、输入门、输出门激活值与细胞状态演化。

In [ ]:
# 准备序列
text = "hello"
inputs = [char_to_idx[ch] for ch in text]
print(f"输入序列：{text}")
print(f"对应索引：{inputs}")

# 存储门激活值
gates_storage = {
    'forget': [],
    'input': [],
    'output': []
}
cell_states_storage = []

# 初始化隐藏状态和细胞状态
h_prev = np.zeros(hidden_size)
C_prev = np.zeros(hidden_size)

# 对序列前向传播
for t, idx in enumerate(inputs):
    # 创建独热编码输入
    x = np.zeros(vocab_size)
    x[idx] = 1.0
    
    # 手动计算各门值，以便记录
    concat = np.concatenate([h_prev, x])
    
    # 遗忘门
    f = lstm.sigmoid(np.dot(lstm.Wf, concat) + lstm.bf)
    gates_storage['forget'].append(f.copy())
    
    # 输入门
    i = lstm.sigmoid(np.dot(lstm.Wi, concat) + lstm.bi)
    gates_storage['input'].append(i.copy())
    
    # 候选细胞状态
    C_tilde = np.tanh(np.dot(lstm.Wc, concat) + lstm.bc)
    
    # 更新细胞状态
    C_prev = f * C_prev + i * C_tilde
    cell_states_storage.append(C_prev.copy())
    
    # 输出门
    o = lstm.sigmoid(np.dot(lstm.Wo, concat) + lstm.bo)
    gates_storage['output'].append(o.copy())
    
    # 更新隐藏状态
    h_prev = o * np.tanh(C_prev)
    
    print(f"\n步骤 {t} ('{text[t]}'):")
    print(f"  遗忘门均值：{f.mean():.3f}（1=保留，0=遗忘）")
    print(f"  输入门均值：{i.mean():.3f}（1=写入，0=忽略）")
    print(f"  输出门均值：{o.mean():.3f}（1=显示，0=隐藏）")

print("\nForward pass complete.")

## 5. 门激活值可视化

使用热力图展示各时间步上哪些隐藏单元活跃，以及细胞状态如何演化。

In [ ]:
# 可视化门激活值
plot_gate_activations(gates_storage, text)

# 可视化细胞状态演化
plot_cell_state_evolution(cell_states_storage, text)

# 分析模式
analyze_gate_patterns(gates_storage, text)

## 6. 在实际文本上训练

在莎士比亚文本上训练 LSTM，观察它如何学习预测下一个字符。任务与第 1 天的基本 RNN 相同，但模型增加了细胞状态机制。

In [ ]:
# 简单训练数据
training_text = """
To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
And by opposing end them.
"""

# 创建词表
chars_train = sorted(list(set(training_text)))
char_to_idx_train = {ch: i for i, ch in enumerate(chars_train)}
idx_to_char_train = {i: ch for i, ch in enumerate(chars_train)}
vocab_size_train = len(chars_train)

print(f"训练文本长度：{len(training_text)} 个字符")
print(f"Vocabulary size: {vocab_size_train}")
print(f"不重复字符：{''.join(chars_train)}")

# 创建 LSTM
lstm_train = LSTM(input_size=vocab_size_train, 
                  hidden_size=64, 
                  output_size=vocab_size_train)

print(f"\n已创建用于训练的 LSTM。")

In [ ]:
# 训练循环
seq_length = 25
learning_rate = 0.001
num_iterations = 1000

losses = []
h_prev = np.zeros(lstm_train.hidden_size)
C_prev = np.zeros(lstm_train.hidden_size)

print('正在训练 LSTM……')
print("=" * 60)

for iteration in range(num_iterations):
    # 随机抽取起点
    start_idx = np.random.randint(0, len(training_text) - seq_length - 1)
    
    # 获取输入与目标序列
    input_seq = training_text[start_idx:start_idx + seq_length]
    target_seq = training_text[start_idx + 1:start_idx + seq_length + 1]
    
    # 转换为索引
    inputs = [char_to_idx_train[ch] for ch in input_seq]
    targets = [char_to_idx_train[ch] for ch in target_seq]
    
    # 前向传播
    loss = lstm_train.forward(inputs, targets, h_prev, C_prev)
    losses.append(loss)
    
    # 反向传播
    dh_next, dC_next = lstm_train.backward()
    
    # 更新权重
    lstm_train.update_weights(learning_rate)
    
    # 更新状态
    h_prev = lstm_train.h_states[-1].copy()
    C_prev = lstm_train.C_states[-1].copy()
    
    # 打印进度
    if iteration % 100 == 0:
        smooth_loss = np.mean(losses[-100:]) if len(losses) >= 100 else np.mean(losses)
        print(f"迭代 {iteration:4d} | 损失：{smooth_loss:.4f}")
        
        # Sample text
        if iteration % 500 == 0:
            sample = lstm_train.sample(idx_to_char_train, char_to_idx_train['T'], 100)
            print(f"样本：{sample[:60]}...")
            print()

print("\nTraining complete.")

In [ ]:
# 绘制训练曲线
plt.figure(figsize=(10, 5))
plt.plot(losses, alpha=0.3, label='原始损失')
# 平滑曲线
window = 50
if len(losses) > window:
    smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
    plt.plot(range(window-1, len(losses)), smoothed, linewidth=2, label='平滑损失', color='red')
plt.xlabel('迭代')
plt.ylabel('损失')
plt.title('LSTM 训练进展')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"最终损失：{losses[-1]:.4f}")
print(f"改善：{(losses[0] - losses[-1]) / losses[0] * 100:.1f}%")

## 7. 温度采样

温度（temperature）控制采样的随机性：

- **较低（0.5）**：更保守，更倾向于选择高概率字符；通常较连贯，但更容易重复。
- **中等（1.0）**：使用模型原始概率分布。
- **较高（1.5）**：更加随机，低概率字符更容易被采到；多样性更高，但连贯性可能下降。

In [ ]:
# 尝试不同温度
temperatures = [0.3, 0.7, 1.0, 1.5]
seed_char = 'T'

print('使用不同温度采样：')
print("=" * 70)

for temp in temperatures:
    sample = lstm_train.sample(idx_to_char_train, 
                               char_to_idx_train[seed_char], 
                               length=150, 
                               temperature=temp)
    print(f"\n温度 = {temp}:")
    print(f"{sample[:120]}...")
    print("-" * 70)

## 8. 总结

**问题：** 基本 RNN 在随时间反向传播时容易发生梯度消失，因而难以学习长程依赖。

**原文的思路：** LSTM 增加细胞状态，提供一条采用加性更新的独立路径。三个门——遗忘门、输入门、输出门——控制写入、保留与暴露哪些信息。

**为什么有效：** 固定门值时，$C_t$ 对 $C_{t-1}$ 的直接路径导数为 $f_t$，无需乘以递归权重矩阵。学习到接近 1 的逐维门值时，有助于保留梯度；这不意味着所有情况下都不会梯度消失。

**原文中的变体：** 窥孔连接、耦合门与 GRU。GRU 将遗忘门与输入门的作用合并为一个更新门。

### 后续步骤

- 尝试 `exercises/` 中的练习，尤其是练习 3 的消融研究，观察移除单个门会发生什么。
- 在相同数据上比较本章与第 1 天基本 RNN 的训练曲线。
- 阅读 Colah 原文中的图示：http://colah.github.io/posts/2015-08-Understanding-LSTMs/